In [1]:
import pandas as pd

df = pd.read_csv("../raw/Vittime.disabilità.csv")

df.head()

,FREQ,Frequenza,REF_AREA,Territorio,DATA_TYPE,Indicatore,SEX,Sesso,Y_PRESENCE_DISABILITY,Presenza di disabilità,TIME_PERIOD,Osservazione,OBS_STATUS,Stato_osservazione
0,A,Annuale,IT,Italia,USERS,Utenti del 1522,M,Maschi,0.0,No,2015.0,1772.0,NaN,NaN
1,A,Annuale,IT,Italia,USERS,Utenti del 1522,M,Maschi,0.0,No,2016.0,1363.0,NaN,NaN
2,A,Annuale,IT,Italia,USERS,Utenti del 1522,M,Maschi,0.0,No,2017.0,1787.0,NaN,NaN
3,A,Annuale,IT,Italia,USERS,Utenti del 1522,M,Maschi,0.0,No,2018.0,2334.0,NaN,NaN
4,A,Annuale,IT,Italia,USERS,Utenti del 1522,M,Maschi,0.0,No,2019.0,2184.0,NaN,NaN


In [2]:
# Per ogni colonna: numero di valori unici + elenco valori

for col in df.columns:
    print("=" * 60)
    print(f"Colonna: {col}")
    print(f"Numero valori unici: {df[col].nunique(dropna=False)}")
    print("Valori unici:")
    print(df[col].unique())
    print("\n")

Colonna: FREQ
Numero valori unici: 89
Valori unici:
['A'
 'A,Annuale,ITC2,\'Valle d"\'Aosta / Vallée d"\'Aoste\',USERS,Utenti del 1522,M,Maschi,0,No,2015,2,,'
 'A,Annuale,ITC2,\'Valle d"\'Aosta / Vallée d"\'Aoste\',USERS,Utenti del 1522,M,Maschi,0,No,2017,1,,'
 'A,Annuale,ITC2,\'Valle d"\'Aosta / Vallée d"\'Aoste\',USERS,Utenti del 1522,M,Maschi,0,No,2018,3,,'
 'A,Annuale,ITC2,\'Valle d"\'Aosta / Vallée d"\'Aoste\',USERS,Utenti del 1522,M,Maschi,0,No,2019,2,,'
 'A,Annuale,ITC2,\'Valle d"\'Aosta / Vallée d"\'Aoste\',USERS,Utenti del 1522,M,Maschi,0,No,2020,4,,'
 'A,Annuale,ITC2,\'Valle d"\'Aosta / Vallée d"\'Aoste\',USERS,Utenti del 1522,M,Maschi,0,No,2021,1,,'
 'A,Annuale,ITC2,\'Valle d"\'Aosta / Vallée d"\'Aoste\',USERS,Utenti del 1522,M,Maschi,0,No,2024,3,,'
 'A,Annuale,ITC2,\'Valle d"\'Aosta / Vallée d"\'Aoste\',USERS,Utenti del 1522,M,Maschi,1,Si,2021,1,,'
 'A,Annuale,ITC2,\'Valle d"\'Aosta / Vallée d"\'Aoste\',USERS,Utenti del 1522,M,Maschi,2,Non indicato,2016,1,,'
 'A,Annuale,ITC

In [3]:
# Rimozione colonne non informative o ridondanti:
# - colonne costanti (un solo valore per tutto il dataset)
# - colonne duplicate sul piano informativo
# - colonne completamente vuote

cols_to_drop = [
    "FREQ",
    "Frequenza",
    "REF_AREA",
    "DATA_TYPE",
    "SEX",
    "Y_PRESENCE_DISABILITY",
    "OBS_STATUS",
    "Stato_osservazione"
]

df = df.drop(columns=cols_to_drop)

# Verifica struttura finale
print("Nuova shape:", df.shape)
print("Colonne rimanenti:", df.columns.tolist())

Nuova shape: (3323, 6)
Colonne rimanenti: ['Territorio', 'Indicatore', 'Sesso', 'Presenza di disabilità', 'TIME_PERIOD', 'Osservazione']


In [4]:
# Standardizzazione nomi colonne:
# - tutto minuscolo
# - nomi più chiari e coerenti

df = df.rename(columns={
    "Territorio": "territorio",
    "Indicatore": "indicatore",
    "Sesso": "sesso",
    "Presenza di disabilità": "presenza_disabilità",
    "TIME_PERIOD": "periodo",
    "Osservazione": "numero_vittime"
})


# Eliminazione delle righe completamente vuote 
df = df.dropna(how="all")

# Stampa delle colonne e dei valori unici che contengono
for col in df.columns:
    print("\nColonna:", col)
    print(df[col].unique())


Colonna: territorio
['Italia' 'Nord-ovest' 'Piemonte' 'Liguria' 'Lombardia' 'Nord-est'
 'Trentino Alto Adige / Südtirol' 'Veneto' 'Friuli-Venezia Giulia'
 'Emilia-Romagna' 'Centro' 'Toscana' 'Umbria' 'Marche' 'Lazio' 'Sud'
 'Abruzzo' 'Molise' 'Campania' 'Puglia' 'Basilicata' 'Calabria' 'Isole'
 'Sicilia' 'Sardegna' 'Non indicato']

Colonna: indicatore
['Utenti del 1522']

Colonna: sesso
['Maschi' 'Femmine' 'Non indicato' 'Totale']

Colonna: presenza_disabilità
['No' 'Si' 'Non indicato' 'Totale']

Colonna: periodo
[2015. 2016. 2017. 2018. 2019. 2020. 2021. 2022. 2023. 2024.]

Colonna: numero_vittime
[ 1772.  1363.  1787. ...  8771. 18639. 22452.]


In [5]:
# Lavora su una copia per evitare SettingWithCopyWarning
df = df.copy()

# Pulizia colonna territorio:
# - uniformazione minuscolo
# - correzione denominazioni incoerenti / verbose
df["territorio"] = (
    df["territorio"]
    .str.lower()
    .str.strip()
    .str.replace("-", " ", regex=False)
)

df["territorio"] = df["territorio"].replace({
    "trentino alto adige / südtirol": "trentino alto adige"
})

# Uniformazione minuscolo per la colonna sesso e presenza_disabilità
df["sesso"] = df["sesso"].str.lower().str.strip()
df["presenza_disabilità"] = df["presenza_disabilità"].str.lower().str.strip()

# Rimozione delle voci aggregate "totale"
# da sesso e presenza_disabilità per evitare doppio conteggio
df = df[
    (df["sesso"] != "totale") &
    (df["presenza_disabilità"] != "totale")
].copy()

# Conversione delle colonne numeriche da float a integer
df["periodo"] = df["periodo"].astype(int)
df["numero_vittime"] = df["numero_vittime"].astype(int)

# Verifica finale post-pulizia (struttura + qualità base)
display(df.head(10))
df.info()

print("\nMissing values per colonna:")
display(df.isna().sum().sort_values(ascending=False))

print("\nNumero valori unici per colonna:")
display(df.nunique(dropna=False).sort_values(ascending=False))

print("\nRighe duplicate:", df.duplicated().sum())

for col in df.columns:
    print("\nColonna:", col)
    if col == "periodo":
        print(sorted(df[col].unique().tolist()))
    else:
        print(df[col].unique())

,territorio,indicatore,sesso,presenza_disabilità,periodo,numero_vittime
0,italia,Utenti del 1522,maschi,no,2015,1772
1,italia,Utenti del 1522,maschi,no,2016,1363
2,italia,Utenti del 1522,maschi,no,2017,1787
3,italia,Utenti del 1522,maschi,no,2018,2334
4,italia,Utenti del 1522,maschi,no,2019,2184
5,italia,Utenti del 1522,maschi,no,2020,2466
6,italia,Utenti del 1522,maschi,no,2021,1883
7,italia,Utenti del 1522,maschi,no,2022,1585
8,italia,Utenti del 1522,maschi,no,2023,419
9,italia,Utenti del 1522,maschi,no,2024,657


<class 'pandas.core.frame.DataFrame'>
Index: 1620 entries, 0 to 3275
Data columns (total 6 columns):
 #   Column               Non-Null Count  Dtype 
---  ------               --------------  ----- 
 0   territorio           1620 non-null   object
 1   indicatore           1620 non-null   object
 2   sesso                1620 non-null   object
 3   presenza_disabilità  1620 non-null   object
 4   periodo              1620 non-null   int64 
 5   numero_vittime       1620 non-null   int64 
dtypes: int64(2), object(4)
memory usage: 88.6+ KB

Missing values per colonna:


territorio             0
indicatore             0
sesso                  0
presenza_disabilità    0
periodo                0
numero_vittime         0
dtype: int64


Numero valori unici per colonna:


numero_vittime         605
territorio              26
periodo                 10
presenza_disabilità      3
sesso                    3
indicatore               1
dtype: int64


Righe duplicate: 0

Colonna: territorio
['italia' 'nord ovest' 'piemonte' 'liguria' 'lombardia' 'nord est'
 'trentino alto adige' 'veneto' 'friuli venezia giulia' 'emilia romagna'
 'centro' 'toscana' 'umbria' 'marche' 'lazio' 'sud' 'abruzzo' 'molise'
 'campania' 'puglia' 'basilicata' 'calabria' 'isole' 'sicilia' 'sardegna'
 'non indicato']

Colonna: indicatore
['Utenti del 1522']

Colonna: sesso
['maschi' 'femmine' 'non indicato']

Colonna: presenza_disabilità
['no' 'si' 'non indicato']

Colonna: periodo
[2015, 2016, 2017, 2018, 2019, 2020, 2021, 2022, 2023, 2024]

Colonna: numero_vittime
[ 1772  1363  1787  2334  2184  2466  1883  1585   419   657   102    83
   114   107    61   167   350   207   441   154   327   244   218   116
    97   770  1775  2056  5294  7467 14948 13897 13783 19753 18040 20657
 16615 14619  7689 13283   357   429   343   398   443  1502  3004  1592
  6282  1878  2199  1618  1371   525   463  6117 12398 12356 27264 37005
    11    14    68    28     1     2  

In [6]:
# Salvataggio dataset pulito

output_path = "../data_clean/vittime.disabilità.csv"

df.to_csv(
    output_path,
    index=False,
    encoding="utf-8-sig"
)

print("File salvato in:", output_path)

File salvato in: ../data_clean/vittime.disabilità.csv
